# DATA 612 — Tier 1 baselines on Criteo (12h windows)

Trains and evaluates the baselines from the proposal on the processed Criteo data in Drive
(`DATA612_data/processed/criteo_w12h`, built by the data-prep notebook).

| Task | Models | Metrics (repo `src/evaluation/metrics.py`) |
|---|---|---|
| **Next-window conversion** | prior (base rate), recency heuristic, logistic regression, gradient boosting, HMM, LSTM | PR-AUC (primary), ROC-AUC, F1, balanced accuracy, BCE, Brier, ECE, top-decile rate / lift / gain |
| **Next-window campaigns** | popularity, user recency (7 days), user history (all past windows) | Precision / Recall / HitRate / NDCG / MAP @ 1, 5, 10 — for all next campaigns **and** for campaigns new to the user |

**Protocols** (from the repo's split columns): `chronological` (future windows) and `user_disjoint`
(unseen users, all windows — the definition used by `scripts/run_baselines.py`). The stricter
`user_disjoint_future` (unseen users *and* future windows) is also available.

**Version 2 (`results/v2_history/`)**: every model also gets the repo's **full-history features**
(`src/data/features.py`: totals since the first impression, windows since first activity / last conversion, …),
so the sequence models see the same information as the gradient-boosting baseline. Version-1 results are kept.

**Design choices**
- Input: the user's last `LOOKBACK` windows (default 14 = 7 days, same as the activity cutoff), built only
  from windows ≤ t. Labels come from window t+1. Leaky columns are never used (the prep pipeline drops them).
- Training uses **all positives + `NEG_RATE` of negatives** (0.18% positive rate). Predicted probabilities
  are corrected back to the true base rate, so calibration metrics stay valid.
- Evaluation always uses the **full** validation and test sets. The F1 threshold is chosen on validation.
- Each finished (protocol, model, seed) is saved to Drive and skipped on re-run, so a disconnect loses little.

**Runtime:** GPU (L4 or A100; T4 also works) **with High-RAM on**.

## 1. Settings

In [1]:
import os, sys, json, time, math, subprocess, importlib.util
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/DATA612_data")
    REPO_DIR  = Path("/content/repo")
else:
    DRIVE_DIR = Path(os.environ.get("DRIVE_DIR", "drive_sim/DATA612_data")).resolve()
    REPO_DIR  = Path(os.environ.get("REPO_DIR", "repo")).resolve()

DATASET    = "criteo_w12h"          # use "criteo_w12h_smoke" for a 2-minute test run
PROTOCOLS  = ["chronological", "user_disjoint"]       # also available: "user_disjoint_future"
MODELS     = ["prior", "recency", "logreg", "gbdt", "hmm", "lstm"]
USE_HISTORY_FEATURES = True         # add the repo's full-history features (src/data/features.py)
HIST_DROP  = ["day_index"]          # absolute day: unseen in the chronological test period, so left out
LOOKBACK   = 14                     # windows of history per prediction (14 x 12h = 7 days)
NEG_RATE   = 0.01                   # share of training negatives kept (all positives kept)
SEEDS      = [0]                    # LSTM seeds; use [0, 1, 2] for the final table (Tier 2)
HMM_STATES = 4
HMM_FIT_SEQS = 20_000               # sequences per class used to fit each HMM
LSTM_EPOCHS, LSTM_PATIENCE, LSTM_BATCH = 8, 2, 2048
CAMPAIGN_EVAL_MAX = 200_000         # test positions (with a non-empty next window) for campaign metrics
CHUNK      = 2_000_000              # rows per prediction chunk (lower if RAM is tight)
FORCE      = False                  # True = recompute results that already exist in Drive
SEED       = 42

DATA_DIR = DRIVE_DIR / "processed" / DATASET
RESULTS_VERSION = "v2_history" if USE_HISTORY_FEATURES else "v1"
OUT_DIR  = DRIVE_DIR / "results" / RESULTS_VERSION / "baselines" / DATASET
(OUT_DIR / "raw").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "models").mkdir(parents=True, exist_ok=True)
print("Data:", DATA_DIR, "\nResults:", OUT_DIR)

Mounted at /content/drive
Data: /content/drive/MyDrive/DATA612_data/processed/criteo_w12h 
Results: /content/drive/MyDrive/DATA612_data/results/v2_history/baselines/criteo_w12h


## 2. Repo (for the shared metrics) and packages

In [2]:
REPO_URL = "https://github.com/Abhyudaya01/criteo-customer-journey-transformers.git"
if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=False)
elif IN_COLAB:
    subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR))

need = [p for p, m in {"pyarrow": "pyarrow", "hmmlearn": "hmmlearn", "psutil": "psutil"}.items()
        if importlib.util.find_spec(m) is None]
if need:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)

import numpy as np, pandas as pd, torch, psutil
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from hmmlearn.hmm import GaussianHMM
from sklearn.metrics import average_precision_score
from src.evaluation.metrics import conversion_metrics, best_f1_threshold, campaign_ranking_metrics

import logging
logging.getLogger("hmmlearn").setLevel(logging.ERROR)   # hides harmless 'not converging' messages
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | RAM {psutil.virtual_memory().total / 1e9:.0f} GB | torch {torch.__version__}")

Device: cuda | RAM 179 GB | torch 2.11.0+cu130


## 3. Load the processed tables

In [3]:
t0 = time.time()
meta = json.loads((DATA_DIR / "meta.json").read_text())
N_WINDOWS = meta["n_windows"]
N_CAMPAIGNS = meta["n_campaigns"]

WINDOW_HOURS = float(meta.get("window_hours", 12))
from src.data.io import shard_ids as data_shards
from src.data.features import build_position_features, feature_matrix, FEATURE_COLUMNS
HIST_COLS = [c for c in FEATURE_COLUMNS if c not in HIST_DROP] if USE_HISTORY_FEATURES else []
hist_keep = [FEATURE_COLUMNS.index(c) for c in HIST_COLS]
pos_parts, uw_parts, h_parts = [], [], []
for s in data_shards(DATA_DIR):                               # shard by shard: tables of a shard share users
    p = pd.read_parquet(DATA_DIR / "positions" / f"part-{s:03d}.parquet",
                        columns=["uid", "window", "label_window", "windows_since_active", "is_active",
                                 "y_conversion", "n_next_campaigns", "time_split", "user_group"])
    u = pd.read_parquet(DATA_DIR / "user_windows" / f"part-{s:03d}.parquet")
    if HIST_COLS:                                             # past-only (as-of merge on windows <= t)
        f = build_position_features(p[["uid", "window", "windows_since_active", "is_active"]], u, WINDOW_HOURS)
        h_parts.append(feature_matrix(f)[:, hist_keep].astype(np.float16))
        del f
    pos_parts.append(p); uw_parts.append(u)
pos = pd.concat(pos_parts, ignore_index=True)
uw = pd.concat(uw_parts, ignore_index=True)
H = np.concatenate(h_parts) if HIST_COLS else np.zeros((len(pos), 0), dtype=np.float16)
del pos_parts, uw_parts, h_parts
print(f"history features: {HIST_COLS if HIST_COLS else 'off'}")

# Contiguous user index and a dense (user, window) -> user_windows row lookup.
uids = np.unique(uw["uid"].to_numpy())
W = 1 << max(6, math.ceil(math.log2(N_WINDOWS)))           # window slots per user (64 for 12h)
uw_u = np.searchsorted(uids, uw["uid"].to_numpy()).astype(np.int64)
dense = np.full(len(uids) * W, len(uw), dtype=np.int32)     # len(uw) = "no record" (zero row)
dense[uw_u * W + uw["window"].to_numpy()] = np.arange(len(uw), dtype=np.int32)

# Per-window features (only past/present information; conversions keyed on their own timestamp).
FEATS = ["log_impressions", "log_clicks", "log_campaigns", "log_cost", "log_conversions", "is_active", "has_record"]
Fw = np.zeros((len(uw) + 1, len(FEATS)), dtype=np.float32)
Fw[:-1, 0] = np.log1p(uw["n_impressions"].to_numpy())
Fw[:-1, 1] = np.log1p(uw["n_clicks"].to_numpy())
Fw[:-1, 2] = np.log1p(uw["n_campaigns"].to_numpy())
Fw[:-1, 3] = np.log1p(uw["cost_sum"].to_numpy().clip(min=0))
Fw[:-1, 4] = np.log1p(uw["n_conversions"].to_numpy())
Fw[:-1, 5] = uw["is_active"].to_numpy()
Fw[:-1, 6] = 1.0

pos_u = np.searchsorted(uids, pos["uid"].to_numpy())
assert (uids[np.minimum(pos_u, len(uids) - 1)] == pos["uid"].to_numpy()).all(), "position user missing from user_windows"
pos_u = pos_u.astype(np.int64)
pos_w = pos["window"].to_numpy().astype(np.int64)
pos_wsa = pos["windows_since_active"].to_numpy().astype(np.float32)
y_all = pos["y_conversion"].to_numpy().astype(np.int8)
SPLIT_CODE = {"train": 0, "val": 1, "test": 2}
def split_codes(col):
    # Compact int8 codes (0 train, 1 val, 2 test); avoids 126M Python strings in memory.
    c = pos[col].astype("category")
    lut = np.array([SPLIT_CODE.get(str(k), -1) for k in c.cat.categories], dtype=np.int8)
    return lut[c.cat.codes.to_numpy()]
time_split = split_codes("time_split")
user_group = split_codes("user_group")
# Group per user (constant within a user), aligned with `uids`; -1 = user has no positions.
first = np.unique(pos_u, return_index=True)
uid_group_code = np.full(len(uids), -1, dtype=np.int8)
uid_group_code[first[0]] = user_group[first[1]]
pos = pos.drop(columns=["time_split", "user_group", "label_window", "window", "windows_since_active",
                        "is_active", "y_conversion"])
del uw_u, first

print(f"{len(pos):,} positions | {int(y_all.sum()):,} positives | {len(uids):,} users | "
      f"{len(uw):,} user-windows | loaded in {time.time() - t0:.0f}s")
if DATASET == "criteo_w12h":
    print("Expected (full 12h data): 125,971,114 positions and 231,004 positives")

history features: ['windows_since_active', 'is_active', 'cur_impressions', 'cur_clicks', 'cur_campaigns', 'cum_impressions', 'cum_clicks', 'cum_conversions', 'cum_active_windows', 'cum_cost', 'windows_since_first', 'windows_since_conversion', 'click_rate', 'slot_of_day']
125,971,114 positions | 231,004 positives | 6,142,256 users | 13,939,200 user-windows | loaded in 259s
Expected (full 12h data): 125,971,114 positions and 231,004 positives


## 4. Features, splits and training sample

In [4]:
LAGS = np.arange(LOOKBACK - 1, -1, -1)                       # oldest ... newest (t)

def split_masks(protocol):
    if protocol == "chronological":
        return {s: time_split == c for s, c in SPLIT_CODE.items()}
    if protocol == "user_disjoint":                          # unseen users, all windows (run_baselines.py)
        return {s: user_group == c for s, c in SPLIT_CODE.items()}
    if protocol == "user_disjoint_future":                   # unseen users and future windows
        return {s: (time_split == c) & (user_group == c) for s, c in SPLIT_CODE.items()}
    raise ValueError(protocol)

rng = np.random.default_rng(SEED)
keep_neg = rng.random(len(pos)) < NEG_RATE                   # one fixed negative sample for all models

# Standardise the history features with statistics from (sampled) chronological-training rows only.
_ref = np.flatnonzero((time_split == SPLIT_CODE["train"]) & keep_neg)
H_MU = H[_ref].astype(np.float32).mean(0) if H.shape[1] else np.zeros(0, np.float32)
H_SD = H[_ref].astype(np.float32).std(0) + 1e-6 if H.shape[1] else np.ones(0, np.float32)

def static_features(idx):
    # Per-position features: recency, calendar, plus the standardised full-history features.
    t = pos_w[idx]
    base = np.stack([pos_wsa[idx] / LOOKBACK, (t % 2).astype(np.float32),
                     np.sin(2 * np.pi * (t % 14) / 14).astype(np.float32),
                     np.cos(2 * np.pi * (t % 14) / 14).astype(np.float32)], axis=1)
    if not H.shape[1]:
        return base
    return np.concatenate([base, (H[idx].astype(np.float32) - H_MU) / H_SD], axis=1).astype(np.float32)
N_STATIC = 4 + H.shape[1]

def build_features(idx):
    # (n, LOOKBACK, F) window history built only from windows <= t, plus (n, N_STATIC) static features.
    u, t = pos_u[idx], pos_w[idx]
    w = t[:, None] - LAGS[None, :]
    rows = dense[u[:, None] * W + np.clip(w, 0, None)]
    rows[w < 0] = len(uw)
    return Fw[rows], static_features(idx)

def train_sample(masks):
    return np.flatnonzero(masks["train"] & ((y_all == 1) | keep_neg))

def to_true_prob(p, r=NEG_RATE):
    # Undo negative subsampling: true odds = sampled odds * r.
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return p * r / (p * r + 1 - p)

for prot in PROTOCOLS:
    m = split_masks(prot)
    tr = train_sample(m)
    print(f"{prot:<22} train sample {len(tr):,} ({int(y_all[tr].sum()):,} pos) | "
          f"val {int(m['val'].sum()):,} ({int(y_all[m['val']].sum()):,} pos) | "
          f"test {int(m['test'].sum()):,} ({int(y_all[m['test']].sum()):,} pos)")

chronological          train sample 1,010,876 (158,834 pos) | val 20,421,186 (34,430 pos) | test 20,121,228 (37,740 pos)
user_disjoint          train sample 1,042,399 (162,013 pos) | val 18,882,244 (34,440 pos) | test 18,878,625 (34,551 pos)


## 5. Models
Every model exposes `fit(train_idx)` and `predict(idx)` → true-scale probabilities.

In [5]:
def predict_chunked(score_fn, idx):
    out = np.empty(len(idx), dtype=np.float64)
    for s in range(0, len(idx), CHUNK):
        X, S = build_features(idx[s:s + CHUNK])
        out[s:s + CHUNK] = score_fn(X, S)
    return out

def platt(scores, y):
    # 1-D logistic calibration of a raw score on the (sampled) training data.
    lr = LogisticRegression(max_iter=1000).fit(scores.reshape(-1, 1), y)
    return lambda s: to_true_prob(lr.predict_proba(s.reshape(-1, 1))[:, 1])

class Prior:
    def fit(self, tr, train_mask):
        self.p = float(y_all[train_mask].mean())             # true base rate (full train split)
    def predict(self, idx):
        return np.full(len(idx), self.p)

class Recency:
    # Recency-weighted activity: recent impressions, clicks and past conversions score higher.
    @staticmethod
    def raw(X, S):
        act = X[:, :, 0] + 2 * X[:, :, 1] + 5 * X[:, :, 4]
        return (act * (0.8 ** LAGS)[None, :]).sum(1)
    def fit(self, tr, train_mask):
        X, S = build_features(tr)
        self.cal = platt(self.raw(X, S), y_all[tr])
    def predict(self, idx):
        return self.cal(predict_chunked(self.raw, idx))

class LogReg:
    @staticmethod
    def flat(X, S):
        return np.concatenate([X.reshape(len(X), -1), X.sum(1), S], axis=1)
    def fit(self, tr, train_mask):
        X, S = build_features(tr)
        Z = self.flat(X, S)
        self.scaler = StandardScaler().fit(Z)
        self.lr = LogisticRegression(max_iter=500, C=1.0).fit(self.scaler.transform(Z), y_all[tr])
    def predict(self, idx):
        f = lambda X, S: self.lr.predict_proba(self.scaler.transform(self.flat(X, S)))[:, 1]
        return to_true_prob(predict_chunked(f, idx))

class GBDT:
    # Same model and settings as src/models/baselines.py (HistGradientBoosting), on the same inputs as LogReg.
    def fit(self, tr, train_mask):
        X, S = build_features(tr)
        self.m = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, max_leaf_nodes=31,
                                                early_stopping=True, validation_fraction=0.1, random_state=SEED)
        self.m.fit(LogReg.flat(X, S), y_all[tr])
    def predict(self, idx):
        return to_true_prob(predict_chunked(lambda X, S: self.m.predict_proba(LogReg.flat(X, S))[:, 1], idx))

def hmm_loglik(model, X, batch=500_000):
    # Vectorised forward algorithm (diagonal Gaussian emissions) -> log-likelihood per sequence.
    mu = torch.tensor(model.means_, dtype=torch.float32, device=DEVICE)
    var = torch.tensor(np.diagonal(model.covars_, axis1=1, axis2=2).copy(), dtype=torch.float32, device=DEVICE)
    logpi = torch.log(torch.tensor(model.startprob_, dtype=torch.float32, device=DEVICE).clamp_min(1e-12))
    logA = torch.log(torch.tensor(model.transmat_, dtype=torch.float32, device=DEVICE).clamp_min(1e-12))
    const = torch.log(2 * math.pi * var).sum(-1)
    out = []
    for s in range(0, len(X), batch):
        x = torch.tensor(X[s:s + batch], device=DEVICE)[:, :, None, :]
        logB = -0.5 * ((((x - mu) ** 2) / var).sum(-1) + const)       # (n, L, K)
        alpha = logpi + logB[:, 0]
        for t in range(1, logB.shape[1]):
            alpha = torch.logsumexp(alpha[:, :, None] + logA[None], dim=1) + logB[:, t]
        out.append(torch.logsumexp(alpha, dim=1).cpu().numpy())
    return np.concatenate(out)

class HMM:
    # Class-conditional HMMs: one fit on converting sequences, one on non-converting ones.
    # Score = log-likelihood ratio, calibrated with Platt scaling.
    def fit(self, tr, train_mask):
        rng_h = np.random.default_rng(SEED)
        self.models = {}
        for label in (1, 0):
            cand = tr[y_all[tr] == label]
            pick = rng_h.choice(cand, size=min(HMM_FIT_SEQS, len(cand)), replace=False)
            X, _ = build_features(pick)
            X = X + rng_h.normal(0, 1e-3, X.shape).astype(np.float32)   # avoids degenerate variances
            m = GaussianHMM(n_components=HMM_STATES, covariance_type="diag", n_iter=50,
                            min_covar=1e-2, random_state=SEED)
            m.fit(X.reshape(-1, X.shape[2]), lengths=[LOOKBACK] * len(X))
            self.models[label] = m
        X, S = build_features(tr)
        self.cal = platt(self.raw(X, S), y_all[tr])
    def raw(self, X, S):
        return hmm_loglik(self.models[1], X) - hmm_loglik(self.models[0], X)
    def predict(self, idx):
        return self.cal(predict_chunked(self.raw, idx))

class LSTMNet(torch.nn.Module):
    def __init__(self, n_feat, n_static, hidden=64):
        super().__init__()
        self.lstm = torch.nn.LSTM(n_feat, hidden, batch_first=True)
        self.head = torch.nn.Sequential(torch.nn.Linear(hidden + n_static, 64), torch.nn.ReLU(),
                                        torch.nn.Dropout(0.1), torch.nn.Linear(64, 1))
    def forward(self, x, s):
        _, (h, _) = self.lstm(x)
        return self.head(torch.cat([h[-1], s], dim=1)).squeeze(-1)

class LSTM:
    def __init__(self, seed, tag):
        self.seed, self.tag = seed, tag
    def _logits(self, X, S, batch=65_536):
        self.net.eval()
        out = []
        with torch.no_grad():
            for s in range(0, len(X), batch):
                out.append(self.net(torch.tensor(X[s:s + batch], device=DEVICE),
                                    torch.tensor(S[s:s + batch], device=DEVICE)).float().cpu().numpy())
        return np.concatenate(out)
    def fit(self, tr, train_mask, val_idx):
        torch.manual_seed(self.seed); np.random.seed(self.seed)
        X, S = build_features(tr)
        y = torch.tensor(y_all[tr], dtype=torch.float32)
        Xt, St = torch.tensor(X), torch.tensor(S)
        # Early stopping on a fixed validation sample (all positives + NEG_RATE negatives).
        vs = val_idx[(y_all[val_idx] == 1) | keep_neg[val_idx]]
        Xv, Sv = build_features(vs)
        self.net = LSTMNet(X.shape[2], S.shape[1]).to(DEVICE)
        opt = torch.optim.Adam(self.net.parameters(), lr=1e-3)
        lossf = torch.nn.BCEWithLogitsLoss()
        best, best_state, bad = -1.0, None, 0
        for ep in range(LSTM_EPOCHS):
            self.net.train()
            perm = torch.randperm(len(y))
            for b in range(0, len(y), LSTM_BATCH):
                j = perm[b:b + LSTM_BATCH]
                loss = lossf(self.net(Xt[j].to(DEVICE), St[j].to(DEVICE)), y[j].to(DEVICE))
                opt.zero_grad(); loss.backward(); opt.step()
            ap = average_precision_score(y_all[vs], self._logits(Xv, Sv))
            print(f"    epoch {ep + 1}: loss {loss.item():.4f} | val-sample PR-AUC {ap:.4f}")
            if ap > best:
                best, bad = ap, 0
                best_state = {k: v.detach().cpu().clone() for k, v in self.net.state_dict().items()}
            else:
                bad += 1
                if bad >= LSTM_PATIENCE:
                    break
        self.net.load_state_dict(best_state)
        torch.save(best_state, OUT_DIR / "models" / f"lstm_{self.tag}.pt")
    def predict(self, idx):
        f = lambda X, S: 1 / (1 + np.exp(-self._logits(X, S)))
        return to_true_prob(predict_chunked(f, idx))

## 6. Train and evaluate: next-window conversion

In [6]:
def evaluate(model_name, protocol, seed):
    path = OUT_DIR / "raw" / f"conversion_{protocol}_{model_name}_seed{seed}.json"
    if path.exists() and not FORCE:
        print(f"  {protocol} / {model_name} / seed {seed}: already done (skipping)")
        return json.loads(path.read_text())
    m = split_masks(protocol)
    tr = train_sample(m)
    val_idx, test_idx = np.flatnonzero(m["val"]), np.flatnonzero(m["test"])
    t0 = time.time()
    if model_name == "lstm":
        model = LSTM(seed, f"{protocol}_seed{seed}")
        model.fit(tr, m["train"], val_idx)
    else:
        model = {"prior": Prior, "recency": Recency, "logreg": LogReg, "gbdt": GBDT, "hmm": HMM}[model_name]()
        model.fit(tr, m["train"])
    fit_s = time.time() - t0
    p_val = model.predict(val_idx)
    thr, _ = best_f1_threshold(y_all[val_idx], p_val)              # chosen on validation only
    p_test = model.predict(test_idx)
    res = {"task": "conversion", "protocol": protocol, "model": model_name, "seed": seed,
           "val_pr_auc": float(average_precision_score(y_all[val_idx], p_val)),
           **conversion_metrics(y_all[test_idx], p_test, threshold=thr),
           "f1_threshold_from_val": thr, "n_test": int(len(test_idx)),
           "n_test_pos": int(y_all[test_idx].sum()), "n_train_sample": int(len(tr)),
           "fit_seconds": round(fit_s, 1), "total_seconds": round(time.time() - t0, 1)}
    path.write_text(json.dumps(res, indent=2))
    print(f"  {protocol} / {model_name} / seed {seed}: test PR-AUC {res['pr_auc']:.4f} | "
          f"ROC-AUC {res['roc_auc']:.4f} | {res['total_seconds']:.0f}s")
    return res

conv_results = []
for prot in PROTOCOLS:
    print(f"\n=== {prot} ===")
    for name in MODELS:
        for seed in (SEEDS if name == "lstm" else [0]):
            conv_results.append(evaluate(name, prot, seed))


=== chronological ===
  chronological / prior / seed 0: test PR-AUC 0.0019 | ROC-AUC 0.5000 | 14s
  chronological / recency / seed 0: test PR-AUC 0.0119 | ROC-AUC 0.7537 | 82s
  chronological / logreg / seed 0: test PR-AUC 0.0193 | ROC-AUC 0.8200 | 159s
  chronological / gbdt / seed 0: test PR-AUC 0.0256 | ROC-AUC 0.8491 | 442s
  chronological / hmm / seed 0: test PR-AUC 0.0055 | ROC-AUC 0.5656 | 166s
    epoch 1: loss 0.2997 | val-sample PR-AUC 0.5137
    epoch 2: loss 0.2909 | val-sample PR-AUC 0.5259
    epoch 3: loss 0.3237 | val-sample PR-AUC 0.5322
    epoch 4: loss 0.3025 | val-sample PR-AUC 0.5369
    epoch 5: loss 0.3082 | val-sample PR-AUC 0.5383
    epoch 6: loss 0.3175 | val-sample PR-AUC 0.5394
    epoch 7: loss 0.3187 | val-sample PR-AUC 0.5407
    epoch 8: loss 0.3292 | val-sample PR-AUC 0.5414
  chronological / lstm / seed 0: test PR-AUC 0.0264 | ROC-AUC 0.8489 | 108s

=== user_disjoint ===
  user_disjoint / prior / seed 0: test PR-AUC 0.0018 | ROC-AUC 0.5000 | 14s
  u

## 7. Train and evaluate: next-window campaigns
Ranking metrics on test positions whose next window has at least one campaign (sampled up to
`CAMPAIGN_EVAL_MAX`), reported twice: for **all** next-window campaigns, and for campaigns **new to the user**
(never seen in any window ≤ t) — about 10% of them on Criteo, where repetition does not help.

In [7]:
ncl = pd.read_parquet(DATA_DIR / "next_campaign_labels", columns=["uid", "window", "campaign_idx"])
ncl_key = np.searchsorted(uids, ncl["uid"].to_numpy()).astype(np.int64) * W + ncl["window"].to_numpy()
_o = np.argsort(ncl_key, kind="stable")
NCL_KEY, NCL_CAMP = ncl_key[_o], ncl["campaign_idx"].to_numpy()[_o].astype(np.int64)
del ncl, ncl_key, _o
wc = pd.read_parquet(DATA_DIR / "window_campaigns", columns=["uid", "window", "campaign_idx", "n_impressions"])

def next_campaigns(idx):
    # (row_in_idx, campaign) pairs for campaigns in window t+1 of each position.
    keys = pos_u[idx] * W + pos_w[idx]
    lo, hi = np.searchsorted(NCL_KEY, keys, "left"), np.searchsorted(NCL_KEY, keys, "right")
    lens = hi - lo
    rows = np.repeat(np.arange(len(idx)), lens)
    j = np.repeat(lo - np.concatenate([[0], np.cumsum(lens)[:-1]]), lens) + np.arange(int(lens.sum()))
    return rows, NCL_CAMP[j]

def campaign_sample(m):
    cand = np.flatnonzero(m["test"] & (pos["n_next_campaigns"].to_numpy() > 0))
    rs = np.random.default_rng(SEED)
    return np.sort(rs.choice(cand, size=min(CAMPAIGN_EVAL_MAX, len(cand)), replace=False)), len(cand)

def campaign_truth(idx):
    y = np.zeros((len(idx), N_CAMPAIGNS), dtype=np.int8)
    r, c = next_campaigns(idx)
    y[r, c] = 1
    return y

def user_history(idx):
    # From window_campaigns rows with window <= t: 7-day recency score, full-history score, "seen ever" mask.
    sample = pd.DataFrame({"row": np.arange(len(idx)), "uid": pos["uid"].to_numpy()[idx], "window": pos_w[idx]})
    h = sample.merge(wc[wc["uid"].isin(sample["uid"])], on="uid", suffixes=("", "_h"))
    lag = (h["window"] - h["window_h"]).to_numpy()
    h, lag = h[lag >= 0], lag[lag >= 0]
    r, c, n = h["row"].to_numpy(), h["campaign_idx"].to_numpy(), h["n_impressions"].to_numpy(np.float32)
    rec7 = np.zeros((len(idx), N_CAMPAIGNS), dtype=np.float32)
    full = np.zeros((len(idx), N_CAMPAIGNS), dtype=np.float32)
    near = lag < LOOKBACK
    np.add.at(rec7, (r[near], c[near]), n[near] * 0.8 ** lag[near])
    np.add.at(full, (r, c), n + 1e-3 * 0.8 ** lag)                 # count first, then recency (like run_baselines)
    return rec7, full, full > 0

def popularity(m, max_rows=2_000_000):
    # Campaign frequency in the next windows of (a sample of) the protocol's training positions.
    tr = np.flatnonzero(m["train"] & (pos["n_next_campaigns"].to_numpy() > 0))
    if len(tr) > max_rows:
        tr = np.random.default_rng(SEED).choice(tr, size=max_rows, replace=False)
    _, c = next_campaigns(tr)
    pop = np.bincount(c, minlength=N_CAMPAIGNS).astype(np.float64)
    return pop / max(pop.sum(), 1)

def ranking_all_and_new(y_true, seen, scores):
    out = {k: v for k, v in campaign_ranking_metrics(y_true, scores, ks=(1, 5, 10)).items()
           if k != "frac_positions_nonempty"}
    y_new = (y_true.astype(bool) & ~seen).astype(np.int8)
    keep = y_new.sum(1) > 0
    new = campaign_ranking_metrics(y_new[keep], scores[keep], ks=(1, 5, 10)) if keep.any() else {}
    out.update({f"new_{k}": v for k, v in new.items() if k != "frac_positions_nonempty"})
    out["frac_positions_with_new_campaign"] = float(keep.mean())
    out["frac_next_campaigns_new"] = float(y_new.sum() / max(y_true.sum(), 1))
    return out

def campaign_eval(protocol):
    path = OUT_DIR / "raw" / f"campaign_{protocol}.json"
    if path.exists() and not FORCE:
        print(f"  {protocol}: already done (skipping)")
        return json.loads(path.read_text())
    m = split_masks(protocol)
    pop = popularity(m)
    idx, n_cand = campaign_sample(m)
    y_true = campaign_truth(idx)
    rec7, full, seen = user_history(idx)
    out = {"task": "campaign", "protocol": protocol, "n_eval_positions": int(len(idx)),
           "frac_test_positions_with_next_campaign": float(n_cand / max(int(m["test"].sum()), 1))}
    scores = {"popularity": np.broadcast_to(pop, y_true.shape).astype(np.float32),
              "user_recency_7d": rec7 + 1e-6 * pop[None, :].astype(np.float32),   # popularity fills gaps
              "user_history": full + 1e-6 * pop[None, :].astype(np.float32)}
    for name, sc in scores.items():
        out[name] = ranking_all_and_new(y_true, seen, np.asarray(sc))
        print(f"  {protocol} / {name}: Recall@10 {out[name]['recall_at_10']:.4f} | "
              f"new-campaign Recall@10 {out[name].get('new_recall_at_10', float('nan')):.4f}")
    print(f"  share of next-window campaigns that are new to the user: {out['popularity']['frac_next_campaigns_new']:.1%}")
    path.write_text(json.dumps(out, indent=2))
    return out

camp_results = [campaign_eval(p) for p in PROTOCOLS]

  chronological / popularity: Recall@10 0.1863 | new-campaign Recall@10 0.1372
  chronological / user_recency_7d: Recall@10 0.8306 | new-campaign Recall@10 0.1275
  chronological / user_history: Recall@10 0.8858 | new-campaign Recall@10 0.1222
  share of next-window campaigns that are new to the user: 13.0%
  user_disjoint / popularity: Recall@10 0.1930 | new-campaign Recall@10 0.1469
  user_disjoint / user_recency_7d: Recall@10 0.8247 | new-campaign Recall@10 0.1409
  user_disjoint / user_history: Recall@10 0.8601 | new-campaign Recall@10 0.1386
  share of next-window campaigns that are new to the user: 16.2%


## 8. Results tables (saved to Drive)

In [8]:
conv = pd.DataFrame(conv_results)
keep = ["protocol", "model", "seed", "pr_auc", "roc_auc", "f1", "balanced_accuracy",
        "binary_cross_entropy", "brier", "expected_calibration_error",
        "top_decile_conversion_rate", "top_decile_lift", "top_decile_gain", "base_rate",
        "val_pr_auc", "f1_threshold_from_val", "n_test", "n_test_pos", "fit_seconds"]
conv = conv[keep].sort_values(["protocol", "pr_auc"], ascending=[True, False])
conv.to_csv(OUT_DIR / "conversion_baselines.csv", index=False)

rows = []
for r in camp_results:
    for name in ("popularity", "user_recency_7d", "user_history"):
        rows.append({"protocol": r["protocol"], "model": name, "n_eval_positions": r["n_eval_positions"], **r[name]})
camp = pd.DataFrame(rows)
camp.to_csv(OUT_DIR / "campaign_baselines.csv", index=False)

import platform
versions = {"python": platform.python_version(), "torch": torch.__version__, "numpy": np.__version__,
            "pandas": pd.__version__, "device": DEVICE, "dataset": DATASET, "lookback": LOOKBACK,
            "neg_rate": NEG_RATE, "seeds": SEEDS, "history_features": HIST_COLS, "protocols": PROTOCOLS}
(OUT_DIR / "run_config.json").write_text(json.dumps(versions, indent=2))

pd.set_option("display.float_format", lambda v: f"{v:.4f}")
print("Next-window conversion (test set):")
display(conv)
print("\nNext-window campaigns (test set, positions with a non-empty next window):")
display(camp[["protocol", "model", "recall_at_10", "hit_rate_at_10", "ndcg_at_10", "new_recall_at_10", "new_ndcg_at_10"]])
print("\nSaved to:", OUT_DIR)

Next-window conversion (test set):


,protocol,model,seed,pr_auc,roc_auc,f1,balanced_accuracy,binary_cross_entropy,brier,expected_calibration_error,top_decile_conversion_rate,top_decile_lift,top_decile_gain,base_rate,val_pr_auc,f1_threshold_from_val,n_test,n_test_pos,fit_seconds
5,chronological,lstm,0,0.0264,0.8489,0.0788,0.5452,0.0117,0.0019,0.0006,0.0103,5.4971,0.5497,0.0019,0.0243,0.0536,20121228,37740,22.8000
3,chronological,gbdt,0,0.0256,0.8491,0.0780,0.5454,0.0117,0.0018,0.0003,0.0103,5.4992,0.5499,0.0019,0.0241,0.0405,20121228,37740,25.2000
2,chronological,logreg,0,0.0193,0.8200,0.0674,0.5466,0.0124,0.0019,0.0009,0.0091,4.8670,0.4867,0.0019,0.0178,0.0582,20121228,37740,16.0000
1,chronological,recency,0,0.0119,0.7537,0.0436,0.5243,0.0137,0.0021,0.0009,0.0080,4.2655,0.4266,0.0019,0.0113,0.0708,20121228,37740,1.9000
4,chronological,hmm,0,0.0055,0.5656,0.0341,0.5150,0.0138,0.0019,0.0013,0.0040,2.1161,0.2116,0.0019,0.0052,0.0034,20121228,37740,93.1000
0,chronological,prior,0,0.0019,0.5000,0.0037,0.5000,0.0137,0.0019,0.0000,0.0019,1.0074,0.1007,0.0019,0.0017,0.0019,20121228,37740,0.3000
9,user_disjoint,gbdt,0,0.0268,0.8484,0.0772,0.5401,0.0114,0.0018,0.0000,0.0101,5.5084,0.5508,0.0018,0.0253,0.0406,18878625,34551,30.0000
11,user_disjoint,lstm,0,0.0265,0.8466,0.0778,0.5421,0.0115,0.0018,0.0002,0.0100,5.4872,0.5487,0.0018,0.0252,0.0335,18878625,34551,17.1000
8,user_disjoint,logreg,0,0.0194,0.8232,0.0674,0.5430,0.0120,0.0019,0.0004,0.0089,4.8566,0.4857,0.0018,0.0186,0.0420,18878625,34551,17.2000
7,user_disjoint,recency,0,0.0130,0.7666,0.0459,0.5255,0.0134,0.0021,0.0010,0.0080,4.3712,0.4371,0.0018,0.0122,0.0745,18878625,34551,2.1000



Next-window campaigns (test set, positions with a non-empty next window):


,protocol,model,recall_at_10,hit_rate_at_10,ndcg_at_10,new_recall_at_10,new_ndcg_at_10
0,chronological,popularity,0.1863,0.1919,0.0947,0.1372,0.0680
1,chronological,user_recency_7d,0.8306,0.8441,0.8000,0.1275,0.0526
2,chronological,user_history,0.8858,0.8954,0.8337,0.1222,0.0488
3,user_disjoint,popularity,0.1930,0.1996,0.0990,0.1469,0.0720
4,user_disjoint,user_recency_7d,0.8247,0.8402,0.7947,0.1409,0.0575
5,user_disjoint,user_history,0.8601,0.8732,0.8192,0.1386,0.0556



Saved to: /content/drive/MyDrive/DATA612_data/results/v2_history/baselines/criteo_w12h


## Notes for the report
- **Primary metric is PR-AUC** (positive rate ≈ 0.18%). A random ranker scores the base rate, which is what
  the `prior` row shows. Every other model should beat it clearly.
- **Negative subsampling** (`NEG_RATE`) is used only for training; probabilities are corrected back, and all
  metrics use the full validation/test sets.
- `user_disjoint` / `user_disjoint_future` train on train users only, so their training sets are smaller.
- Gradient boosting here uses the same inputs as logistic regression (7-day window + full-history features), so it
  is slightly richer than `scripts/run_baselines.py`'s GBDT (full-history features only).
- `new_*` campaign metrics: next-window campaigns the user had never seen before window t+1.
- For the final table (Tier 2), set `SEEDS = [0, 1, 2]` and add bootstrap confidence intervals.